# Computer Networks · Week 3 — Applications notebook (Google Colab)
### The application layer: sockets, the Web and HTTP/1.1

Companion notebook to the **Week 3 teaching pack** (Lecture 3, Seminar 3, Laboratory 3 Linux, Laboratory 3 Windows).

| Part | In the teaching pack | In this notebook |
|---|---|---|
| **A** | Lecture 3 | Client–server vs P2P in numbers; sockets, ports and 4-tuples observed live; TCP vs UDP message boundaries; URL anatomy; an HTTP page-load calculator; a small **HTTP/1.1 server and client written from raw sockets** that implement Content-Length, chunked bodies, persistent connections and conditional GET; network byte order |
| **B** | Seminar 3 | Problems 1–10 computed and *run*: a relay that cuts a TCP stream at random places, status codes produced by a real server, live 4-tuples, and the file-upload protocol of Problem 10 implemented end to end |
| **C** | Laboratory 3 (Linux) | All C programs of the sheet (`netio.h`, `echo_client.c`, `echo_server.c`, `upper_server.c`, `blast.c`, `msg_client.c`, `msg_server.c`, plus `file_client.c` and `broken_pipe.c`) written, compiled with `make` and run on the Colab VM, Tasks 1–8, and the Python versions |
| **D** | Laboratory 3 (Windows) | The Winsock programs **cross-compiled for Windows** with MinGW-w64 (download the `.exe` files and run them on your PC), the Winsock/POSIX differences, Winsock error codes, and optionally the PowerShell part of the sheet |

**How to use it.** Run the cells in order (`Runtime → Run all` works). Servers are started in the background by the notebook, so you never need a second terminal. Cells marked ✍️ are for your answers to Q1–Q16, so this notebook can be your report (`File → Download → .ipynb`).

## 0 · Setup

In [ ]:
# ---------------------------------------------------------------
#  Helper functions used throughout the notebook. Run this cell first.
# ---------------------------------------------------------------
import os, re, sys, time, math, json, socket, shutil, subprocess, threading, ipaddress
from collections import deque, Counter

IN_COLAB = 'google.colab' in sys.modules
SUDO = '' if os.geteuid() == 0 else 'sudo '

def sh(cmd, timeout=60, show=True):
    """Run a shell command, print its output (stdout+stderr), never raise."""
    try:
        r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
        out = (r.stdout or '') + (r.stderr or '')
    except subprocess.TimeoutExpired as e:
        o = e.stdout or ''
        out = (o.decode(errors='replace') if isinstance(o, bytes) else o) + f'\n[stopped after {timeout} s]'
    if show:
        print(f'$ {cmd}\n{out.rstrip()}\n')
    return out

def have(tool):
    return shutil.which(tool) is not None

def default_route():
    """(interface, gateway) used to reach the Internet."""
    out = sh('ip route get 1.1.1.1', show=False)
    dev = re.search(r'\bdev (\S+)', out)
    via = re.search(r'\bvia (\S+)', out)
    src = re.search(r'\bsrc (\S+)', out)
    return (dev.group(1) if dev else 'any', via.group(1) if via else None, src.group(1) if src else None)

def resolve(host):
    try:
        return socket.gethostbyname(host)
    except Exception as e:
        print(f'cannot resolve {host}: {e}')
        return None

def tcp_rtt(host, port=443, n=5, timeout=3):
    """Measure RTT as the time a TCP connect() takes (one SYN / SYN-ACK exchange).
    Works where ICMP ping is blocked (as on many cloud VMs)."""
    ip = resolve(host)
    if not ip:
        return []
    rtts = []
    for _ in range(n):
        s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        s.settimeout(timeout)
        t0 = time.perf_counter()
        try:
            s.connect((ip, port))
            rtts.append((time.perf_counter() - t0) * 1000)
        except Exception as e:
            print(f'  connect to {ip}:{port} failed: {e}')
        finally:
            s.close()
        time.sleep(0.2)
    if rtts:
        print(f'TCP-connect RTT to {host} ({ip}:{port}): min/avg/max = '
              f'{min(rtts):.2f}/{sum(rtts)/len(rtts):.2f}/{max(rtts):.2f} ms  ({len(rtts)}/{n} answered)')
    return rtts

def fmt_time(seconds):
    """Pretty-print a duration."""
    s = abs(seconds)
    if s < 1e-6: return f'{seconds*1e9:.3g} ns'
    if s < 1e-3: return f'{seconds*1e6:.4g} µs'
    if s < 1:    return f'{seconds*1e3:.4g} ms'
    if s < 120:  return f'{seconds:.4g} s'
    if s < 7200: return f'{seconds/60:.4g} min'
    if s < 2*86400: return f'{seconds/3600:.4g} h'
    return f'{seconds/86400:.4g} days'

def check(name, got, expected, rel=0.02):
    ok = math.isclose(got, expected, rel_tol=rel, abs_tol=1e-12)
    print(f"{'✅' if ok else '❌'} {name}: computed {got:.6g}, expected ≈ {expected:.6g}")
    return ok

print('Helpers loaded. Running in Colab:', IN_COLAB, '| root:', os.geteuid() == 0)

In [ ]:
# Compiler, cross-compiler for Windows, netcat. About 1–2 minutes on Colab.
sh(f'{SUDO}apt-get -qq update > /dev/null 2>&1; DEBIAN_FRONTEND=noninteractive {SUDO}apt-get -qq install -y '
   'build-essential mingw-w64 netcat-openbsd iproute2 curl > /dev/null 2>&1; echo apt done', timeout=900)
for t in ['gcc', 'make', 'x86_64-w64-mingw32-gcc', 'nc', 'ss', 'curl']:
    print(f'{t:24s}', '✅' if have(t) else '❌ missing')

In [ ]:
import struct, random, signal, select, urllib.parse, email.utils, datetime
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.figsize': (9, 4.5), 'axes.grid': True, 'grid.alpha': .3})

LAB = os.path.abspath('lab3')
for d in ('lab3', 'lab3/python', 'lab3/win'):
    os.makedirs(d, exist_ok=True)

class Bg:
    """A program running in the background (like a second terminal), with its output collected."""
    ALL = []
    def __init__(self, cmd, cwd=LAB, settle=0.5):
        self.cmd = cmd
        self.p = subprocess.Popen('exec ' + cmd, shell=True, cwd=cwd, stdin=subprocess.PIPE,
                                  stdout=subprocess.PIPE, stderr=subprocess.STDOUT, start_new_session=True)
        self.buf, self.mark = bytearray(), 0
        threading.Thread(target=self._reader, daemon=True).start()
        Bg.ALL.append(self)
        time.sleep(settle)
    def _reader(self):
        fd = self.p.stdout.fileno()
        while True:
            try:
                chunk = os.read(fd, 4096)
            except OSError:
                break
            if not chunk:
                break
            self.buf += chunk
    @property
    def pid(self):
        return self.p.pid
    def write(self, text):                     # type into the program
        self.p.stdin.write(text.encode()); self.p.stdin.flush()
    def eof(self):                             # like pressing Ctrl+D
        self.p.stdin.close()
    def output(self):
        return self.buf.decode(errors='replace')
    def new_output(self):                      # output since the last call
        out = self.buf[self.mark:].decode(errors='replace'); self.mark = len(self.buf); return out
    def running(self):
        return self.p.poll() is None
    def stop(self, sig=signal.SIGTERM):        # like pressing Ctrl+C (SIGINT) or kill (SIGTERM)
        if self.running():
            try:
                os.killpg(self.p.pid, sig); self.p.wait(timeout=3)
            except Exception:
                try: os.killpg(self.p.pid, signal.SIGKILL)
                except Exception: pass
        return self.p.returncode
    @classmethod
    def stop_all(cls):
        for b in cls.ALL: b.stop()
        cls.ALL.clear()

def show(title, text):
    print(f'── {title} ' + '─' * max(0, 60 - len(title)))
    print(text.rstrip() if text.strip() else '(nothing)')

---
# Part A · Lecture 3 — the lecture in code

## A.1 Application architectures: client–server vs peer-to-peer
How long does it take to give one file of size **F** to **N** hosts? With a server of upload rate *uₛ* and clients of download rate *d*:
client–server `D = max(N·F/uₛ, F/d)`; peer-to-peer, where every peer also uploads at *u*: `D = max(F/uₛ, F/d, N·F/(uₛ + N·u))`. P2P is **self-scaling**: every new peer brings demand *and* capacity.

In [ ]:
F, us, u, d = 1e9, 30e6, 2e6, 20e6        # 1 Gbit file, server 30 Mbit/s up, peers 2 Mbit/s up, 20 Mbit/s down
N = np.arange(1, 1001)
cs = np.maximum(N * F / us, F / d)
p2p = np.maximum.reduce([np.full(N.shape, F / us), np.full(N.shape, F / d), N * F / (us + N * u)])
plt.plot(N, cs / 3600, label='client–server'); plt.plot(N, p2p / 3600, label='peer-to-peer')
plt.xlabel('number of hosts N'); plt.ylabel('time to distribute the file (hours)'); plt.legend()
plt.title('Client–server grows linearly; P2P levels off'); plt.show()
for n in (10, 100, 1000):
    print(f'N = {n:4d}: client–server {cs[n-1]/60:7.1f} min   P2P {p2p[n-1]/60:6.1f} min')

## A.2 Processes, ports and the 4-tuple — observed live
One server process listens on one port; each client gets an **ephemeral** port from its operating system; the server tells connections apart by the 4-tuple *(source IP, source port, destination IP, destination port)*.

In [ ]:
lo, hi = open('/proc/sys/net/ipv4/ip_local_port_range').read().split()
print(f'Ephemeral ports on this Linux host: {lo}–{hi}\n')

ls_ = socket.socket(); ls_.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
ls_.bind(('127.0.0.1', 0)); ls_.listen(8); sport = ls_.getsockname()[1]
clients = [socket.create_connection(('127.0.0.1', sport)) for _ in range(3)]
conns = [ls_.accept()[0] for _ in range(3)]
print(f'Server listening on 127.0.0.1:{sport}; three connections, as the server sees them:')
for c in conns:
    (rip, rport), (lip, lport) = c.getpeername(), c.getsockname()
    print(f'   4-tuple: ({rip}, {rport}, {lip}, {lport})')
sh(f'ss -tn | grep -E "State|:{sport}"')
for s in clients + conns + [ls_]: s.close()

## A.3 What applications need, and what TCP and UDP give
The decisive difference for programmers: **UDP keeps message boundaries, TCP does not.** Two `send()` calls of 5 bytes each:

In [ ]:
# UDP: one send = one datagram = one recvfrom
r = socket.socket(socket.AF_INET, socket.SOCK_DGRAM); r.bind(('127.0.0.1', 0))
s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
for m in (b'HELLO', b'WORLD'): s.sendto(m, r.getsockname())
time.sleep(0.2)
print('UDP receiver got:', [r.recvfrom(100)[0] for _ in range(2)])
r.close(); s.close()

# TCP: a stream of bytes; the receiver gets whatever has arrived
ls_ = socket.socket(); ls_.bind(('127.0.0.1', 0)); ls_.listen(1)
c = socket.create_connection(ls_.getsockname()); a, _ = ls_.accept()
c.send(b'HELLO'); c.send(b'WORLD')
time.sleep(0.2)
print('TCP receiver got:', a.recv(100), '  ← one recv, both messages glued together')
for x in (c, a, ls_): x.close()

In [ ]:
NEEDS = {  # application: (data loss, throughput, time-sensitive)
    'File transfer, download': ('no loss', 'elastic', 'no'),
    'E-mail': ('no loss', 'elastic', 'no'),
    'Web pages': ('no loss', 'elastic', 'somewhat (seconds)'),
    'Voice and video calls': ('loss-tolerant', 'audio kbit/s; video Mbit/s', 'yes, ~100 ms'),
    'Streaming stored video': ('loss-tolerant', 'Mbit/s', 'yes, a few seconds'),
    'Interactive games': ('loss-tolerant', 'kbit/s and up', 'yes, tens of ms'),
    'Text messaging': ('no loss', 'elastic', 'yes and no'),
}
def suggest_transport(loss, timing):
    if loss == 'no loss': return 'TCP (+ TLS if it must be secure)'
    return 'UDP (+ RTP/QUIC-style mechanisms on top)' if timing.startswith('yes') else 'TCP or UDP'
print(f'{"application":24s} {"loss":14s} {"throughput":27s} {"timing":20s} → transport')
for app, (loss, thr, tim) in NEEDS.items():
    print(f'{app:24s} {loss:14s} {thr:27s} {tim:20s} → {suggest_transport(loss, tim)}')

## A.4 Objects and URLs
`https://www.example.org:8443/courses/networks/week3.html?lang=en#exercises` — the **fragment** is used only by the browser and is never sent to the server.

In [ ]:
url = 'https://www.example.org:8443/courses/networks/week3.html?lang=en#exercises'
u = urllib.parse.urlsplit(url)
for part in ('scheme', 'hostname', 'port', 'path', 'query', 'fragment'):
    print(f'{part:9s} {getattr(u, part)}')
print('default port for http  :', urllib.parse.urlsplit('http://example.org/').port or 80)
target = u.path + ('?' + u.query if u.query else '')
print('\nWhat the browser actually sends:')
print(f'GET {target} HTTP/1.1\r\nHost: {u.hostname}:{u.port}\r\n')

## A.5 HTTP connections and their cost in round trips
* **non-persistent**: `2 RTT + transmission` per object (handshake + request);
* **persistent**: one handshake, then `1 RTT + transmission` per object;
* **pipelining / HTTP/2**: all requests at once, one RTT, then the objects back to back;
* **parallel**: several non-persistent connections at once (their transmissions may share one link).

In [ ]:
def page_load(n_objects, rtt, t_base=0.0, t_obj=0.0, mode='nonpersistent', parallel=1, shared_link=False):
    """Time to fetch a base file plus n_objects referenced objects from one server."""
    base = 2 * rtt + t_base                                   # first connection: handshake + request
    if mode == 'nonpersistent':
        if parallel <= 1:
            return base + n_objects * (2 * rtt + t_obj)
        rounds, left, t = 0, n_objects, base
        while left > 0:
            k = min(parallel, left)
            t += 2 * rtt + (k * t_obj if shared_link else t_obj)
            left -= k
        return t
    if mode == 'persistent':
        return base + n_objects * (rtt + t_obj)
    if mode == 'pipelined':
        return base + (rtt + n_objects * t_obj if n_objects else 0)
    raise ValueError(mode)

RTT = 0.050
print('Worked example — base file + 10 small images, RTT 50 ms, transmission ignored')
check('non-persistent, one at a time (ms)', page_load(10, RTT) * 1e3, 1100)
check('non-persistent, 5 parallel (ms)', page_load(10, RTT, parallel=5) * 1e3, 300)
check('persistent (ms)', page_load(10, RTT, mode='persistent') * 1e3, 600)
check('persistent + pipelining / HTTP/2 (ms)', page_load(10, RTT, mode='pipelined') * 1e3, 150)

In [ ]:
# Figure 1 of the lecture: base page + 2 images, in RTTs
def timeline(ax, title, steps):
    y = 0
    for label, start, length, color in steps:
        ax.barh(y, length, left=start, color=color, edgecolor='k'); ax.text(start + length / 2, y, label, ha='center', va='center', fontsize=8)
        y += 1
    ax.set_yticks([]); ax.set_xlim(0, 6.2); ax.set_title(title, fontsize=10); ax.invert_yaxis(); ax.set_xlabel('time (RTT)')

HS, RQ = '#ffd9a8', '#b8d8ff'
fig, axs = plt.subplots(1, 3, figsize=(13, 3.2))
timeline(axs[0], 'Non-persistent: 6 RTT', [('SYN', 0, 1, HS), ('GET base', 1, 1, RQ), ('SYN', 2, 1, HS), ('GET img1', 3, 1, RQ), ('SYN', 4, 1, HS), ('GET img2', 5, 1, RQ)])
timeline(axs[1], 'Persistent: 4 RTT', [('SYN', 0, 1, HS), ('GET base', 1, 1, RQ), ('GET img1', 2, 1, RQ), ('GET img2', 3, 1, RQ)])
timeline(axs[2], 'Pipelined / HTTP/2: 3 RTT', [('SYN', 0, 1, HS), ('GET base', 1, 1, RQ), ('GET img1 + img2', 2, 1, RQ)])
plt.tight_layout(); plt.show()

objs = np.arange(0, 61)
for mode, kw in [('non-persistent', dict(mode='nonpersistent')), ('non-persistent ×6 parallel', dict(mode='nonpersistent', parallel=6)),
                 ('persistent', dict(mode='persistent')), ('pipelined / HTTP/2', dict(mode='pipelined'))]:
    plt.plot(objs, [page_load(n, RTT, **kw) * 1e3 for n in objs], label=mode)
plt.xlabel('objects referenced by the page'); plt.ylabel('page-load time (ms), RTT = 50 ms'); plt.legend()
plt.title('With many small objects, round trips dominate'); plt.show()

## A.6 HTTP/1.1 messages — a server and a client from raw sockets
To see the protocol and not a library, below is a **small HTTP/1.1 server written directly on sockets** (about 100 lines). It implements: persistent connections (and `Connection: close`), `Content-Length`, `Transfer-Encoding: chunked`, conditional GET (`If-Modified-Since` → `304`), redirection, authentication errors, `405` and `400`, and `POST` with an idempotency key. It counts TCP connections and requests, so we can *see* persistence.

In [ ]:
LAST_MOD = 'Fri, 16 Oct 2026 14:12:30 GMT'
LAST_MOD_TS = email.utils.parsedate_to_datetime(LAST_MOD)

class MiniHTTPServer(threading.Thread):
    def __init__(self, port=0):
        super().__init__(daemon=True)
        self.ls = socket.socket(); self.ls.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        self.ls.bind(('127.0.0.1', port)); self.ls.listen(16)
        self.port = self.ls.getsockname()[1]
        self.connections = self.requests = 0
        self.orders, self.seen_keys = [], {}
        self.start()

    def run(self):
        while True:
            try:
                c, _ = self.ls.accept()
            except OSError:
                return
            self.connections += 1
            threading.Thread(target=self.serve, args=(c,), daemon=True).start()

    def serve(self, c):
        rf = c.makefile('rb')
        with c:
            while True:
                line = rf.readline()
                if not line:
                    return                                          # client closed
                parts = line.decode('latin-1').rstrip('\r\n').split(' ')
                headers = {}
                while (h := rf.readline()) not in (b'\r\n', b'\n', b''):   # header ends at the empty line
                    k, _, v = h.decode('latin-1').partition(':')
                    headers[k.strip().lower()] = v.strip()
                body = rf.read(int(headers.get('content-length', 0)))      # body framed by Content-Length
                self.requests += 1
                if len(parts) != 3 or not parts[2].startswith('HTTP/'):
                    self.reply(c, 400, 'Bad Request', b'bad request line\n', close=True); return
                method, target, version = parts
                close = headers.get('connection', '').lower() == 'close' or version == 'HTTP/1.0'
                self.route(c, method, target, headers, body, close)
                if close:
                    return

    def reply(self, c, code, reason, body=b'', headers=None, close=False, chunks=None):
        h = {'Date': email.utils.formatdate(usegmt=True), 'Server': 'mini-http/0.1'}
        h.update(headers or {})
        if chunks is not None:
            h['Transfer-Encoding'] = 'chunked'
        elif code != 304:
            h['Content-Length'] = str(len(body))
        if close:
            h['Connection'] = 'close'
        head = f'HTTP/1.1 {code} {reason}\r\n' + ''.join(f'{k}: {v}\r\n' for k, v in h.items()) + '\r\n'
        if chunks is not None:
            body = b''.join(f'{len(ch):x}\r\n'.encode() + ch + b'\r\n' for ch in chunks) + b'0\r\n\r\n'
        c.sendall(head.encode('latin-1') + (b'' if code == 304 else body))

    def route(self, c, method, target, headers, body, close):
        KNOWN = {'GET', 'HEAD', 'POST', 'PUT', 'DELETE', 'PATCH', 'OPTIONS'}
        if method not in KNOWN:
            return self.reply(c, 501, 'Not Implemented', b'unknown method\n', close=close)
        if method == 'DELETE':
            return self.reply(c, 405, 'Method Not Allowed', b'pages are never deleted\n', {'Allow': 'GET, HEAD, POST'}, close)
        if target == '/old':
            return self.reply(c, 301, 'Moved Permanently', b'', {'Location': 'https://cn.example.edu/'}, close)
        if target == '/admin':
            who = headers.get('authorization')
            if not who:
                return self.reply(c, 401, 'Unauthorized', b'log in first\n', {'WWW-Authenticate': 'Basic realm="cn"'}, close)
            if who != 'teacher':
                return self.reply(c, 403, 'Forbidden', b'teachers only\n', close=close)
            return self.reply(c, 200, 'OK', b'admin page\n', close=close)
        if target == '/crash':
            return self.reply(c, 500, 'Internal Server Error', b'the application crashed\n', close=close)
        if target == '/stream':                                     # body of unknown length → chunked
            return self.reply(c, 200, 'OK', headers={'Content-Type': 'text/plain'}, chunks=[b'Network', b's, week 3'], close=close)
        if target == '/files/notes.pdf':                            # conditional GET
            ims = headers.get('if-modified-since')
            meta = {'Last-Modified': LAST_MOD, 'ETag': '"5f3a-1e2b4"'}
            if ims and email.utils.parsedate_to_datetime(ims) >= LAST_MOD_TS:
                return self.reply(c, 304, 'Not Modified', headers=meta, close=close)
            return self.reply(c, 200, 'OK', b'%PDF-1.7 ... (pretend this is a PDF)\n', {**meta, 'Content-Type': 'application/pdf'}, close)
        if target == '/orders' and method == 'POST':                # POST with an idempotency key
            key = headers.get('idempotency-key')
            if key and key in self.seen_keys:
                return self.reply(c, 200, 'OK', self.seen_keys[key], close=close)
            self.orders.append(body)
            msg = f'order {len(self.orders)} created\n'.encode()
            if key: self.seen_keys[key] = msg
            return self.reply(c, 201, 'Created', msg, {'Location': f'/orders/{len(self.orders)}'}, close)
        if target.startswith('/hello') or target == '/':
            return self.reply(c, 200, 'OK', b'<!doctype html><title>hi</title><p>Hello, week 3!</p>\n',
                              {'Content-Type': 'text/html; charset=utf-8'}, close)
        return self.reply(c, 404, 'Not Found', b'no such object\n', close=close)

    def shutdown(self):
        self.ls.close()

web = MiniHTTPServer()
print('mini HTTP server on 127.0.0.1:%d' % web.port)

### The client side: where does a message end?
The reader below applies exactly the three rules of the lecture: the **header** ends at the first empty line; a **body** is `Content-Length` bytes long, or a sequence of **chunks** ended by a chunk of size 0 (or, failing both, ends when the connection closes).

In [ ]:
def decode_chunked(raw: bytes):
    """Decode a chunked body. Returns (body, number of chunks)."""
    body, pos, n = b'', 0, 0
    while True:
        eol = raw.index(b'\r\n', pos)
        size = int(raw[pos:eol].split(b';')[0], 16)               # size in hexadecimal
        pos = eol + 2
        if size == 0:
            return body, n
        body += raw[pos:pos + size]; pos += size + 2; n += 1       # skip the CR LF after the data

def read_response(rf, method='GET'):
    """Read ONE HTTP/1.1 response from a buffered socket file. Returns (status, headers, body, framing)."""
    status = rf.readline().decode('latin-1').rstrip('\r\n')
    headers = {}
    while (h := rf.readline()) not in (b'\r\n', b'\n', b''):
        k, _, v = h.decode('latin-1').partition(':'); headers[k.strip().lower()] = v.strip()
    code_ = int(status.split()[1])
    if method == 'HEAD' or code_ in (204, 304) or 100 <= code_ < 200:
        return status, headers, b'', 'no body'
    if headers.get('transfer-encoding', '').lower() == 'chunked':
        raw = b''
        while True:
            line = rf.readline(); raw += line
            size = int(line.split(b';')[0], 16)
            if size == 0:
                raw += rf.readline(); break
            raw += rf.read(size + 2)
        return status, headers, decode_chunked(raw)[0], 'chunked'
    if 'content-length' in headers:
        return status, headers, rf.read(int(headers['content-length'])), 'Content-Length'
    return status, headers, rf.read(), 'until the connection closed'

def request(method, target, host='localhost', headers=None, body=b'', close=False):
    h = {'Host': host, 'User-Agent': 'course-notebook/1.0', **(headers or {})}
    if body: h['Content-Length'] = str(len(body))
    if close: h['Connection'] = 'close'
    return (f'{method} {target} HTTP/1.1\r\n' + ''.join(f'{k}: {v}\r\n' for k, v in h.items()) + '\r\n').encode() + body

def exchange(sock, rf, method, target, verbose=True, **kw):
    msg = request(method, target, **kw)
    sock.sendall(msg)
    status, headers, body, framing = read_response(rf, method)
    if verbose:
        print('>>> ' + msg.decode('latin-1').replace('\r\n', '\n>>> ').rstrip('> \n'))
        print('<<< ' + status); [print(f'<<< {k}: {v}') for k, v in headers.items()]
        print(f'<<< (body: {len(body)} bytes, framed by {framing}) {body[:60]!r}\n')
    return status, headers, body

c0, r0 = web.connections, web.requests
s = socket.create_connection(('127.0.0.1', web.port)); rf = s.makefile('rb')
exchange(s, rf, 'GET', '/hello')
exchange(s, rf, 'GET', '/stream')
exchange(s, rf, 'GET', '/files/notes.pdf', close=True)
rf.close(); s.close()      # close the file object too, or the connection stays open
print(f'Server statistics: {web.requests - r0} requests over {web.connections - c0} TCP connection(s) → persistent connection')

In [ ]:
# HTTP/1.0 behaviour: one connection per object (non-persistent)
c0 = web.connections
for target in ('/hello', '/hello?img=1', '/hello?img=2'):
    with socket.create_connection(('127.0.0.1', web.port)) as s:
        s.sendall(f'GET {target} HTTP/1.0\r\nHost: localhost\r\n\r\n'.encode())
        print(f'{target:14s} →', s.makefile('rb').readline().decode().strip())
print(f'3 objects needed {web.connections - c0} TCP connections')

In [ ]:
METHODS = [('GET', 'retrieve the object', True, True), ('HEAD', 'like GET, header only', True, True),
           ('POST', 'submit data to be processed', False, False), ('PUT', 'create or replace the object', False, True),
           ('DELETE', 'delete the object', False, True), ('PATCH', 'modify part of an object', False, False),
           ('OPTIONS', 'which methods are supported', True, True)]
print(f'{"method":8s} {"meaning":32s} {"safe":5s} {"idempotent"}')
for m, what, safe, idem in METHODS:
    print(f'{m:8s} {what:32s} {"yes" if safe else "no":5s} {"yes" if idem else "no"}')

### A real server
The same reader works against a real web server on the Internet (if the Colab VM can reach it).

In [ ]:
try:
    with socket.create_connection(('example.org', 80), timeout=8) as s:
        rf = s.makefile('rb')
        exchange(s, rf, 'GET', '/', host='example.org')
        exchange(s, rf, 'HEAD', '/', host='example.org', close=True)
except OSError as e:
    print('example.org not reachable from here:', e)

## A.7 The socket API, step by step, and network byte order
Python's `socket` module is a thin layer over the same system calls as C: `socket → bind → listen → accept → recv/send → close` on the server, `socket → connect → send/recv → close` on the client. One more fact: integers on the wire are **big-endian**. Forgetting `htons` on an x86 (little-endian) machine makes the server listen on the wrong port:

In [ ]:
print('this machine is', sys.byteorder, 'endian')
for port in (8080, 5000):
    wrong = struct.unpack('!H', struct.pack('<H', port))[0]    # bytes in host order read as network order
    print(f'port {port} = 0x{port:04X} → without htons the bytes {struct.pack("<H", port).hex(" ")} are read as 0x{wrong:04X} = {wrong}')
print('socket.htons(8080) =', socket.htons(8080))

## A.8 End-of-lecture exercises, checked by code

In [ ]:
RTT, T = 0.080, 0.010
print('Exercise 1 — base + 4 images, RTT 80 ms, 10 ms per object')
check('(a) non-persistent (ms)', page_load(4, RTT, T, T) * 1e3, 850)
check('(b) persistent (ms)', page_load(4, RTT, T, T, 'persistent') * 1e3, 530)
check('(c) pipelined (ms)', page_load(4, RTT, T, T, 'pipelined') * 1e3, 290)
print('Exercise 2 — 4 parallel non-persistent connections')
check('page-load time (ms)', page_load(4, RTT, T, T, parallel=4) * 1e3, 340)
print('Exercise 3 — chunked body')
body, n = decode_chunked(b'5\r\nHello\r\n8\r\n, world!\r\n0\r\n\r\n')
print(f'   {body!r}, {len(body)} bytes in {n} chunks'); check('length', len(body), 13)
print('Exercise 4 — one send() is NOT guaranteed to arrive as one recv(), whatever its size (see B.6)')
print('Exercise 5 — sin_port = 8080 without htons')
check('real port', struct.unpack('!H', struct.pack('<H', 8080))[0], 36895)

**Review questions**: (1) why new applications need no change to routers · (2) client and server in BitTorrent · (3) 10 000 connections to port 443: how is a segment delivered? · (4) an application for which UDP is better · (5) how HTTPS protects a connection, at which layer · (6) the URL part never sent · (7) why `Host` is mandatory · (8) repeating GET vs POST · (9) two ways to find the end of a body · (10) what `accept()` returns.

**✍️ Your answers** (double-click to edit)

- **R1:** 
- **R2:** 
- **R3:** 
- **R4:** 
- **R5:** 
- **R6:** 
- **R7:** 
- **R8:** 
- **R9:** 
- **R10:** 

---
# Part B · Seminar 3 — HTTP timing, messages and byte streams

## B.1 Problem 1 — which transport?

In [ ]:
P1 = {'a': ('Remote login (SSH)', 'TCP'),
      'b': ('Racing game, positions 30×/s', 'UDP'),
      'c': ('2 GB backup upload', 'TCP'),
      'd': ('Video call between two phones', 'UDP+RTP'),
      'e': ('DNS query', 'UDP'),
      'f': ('Sensor reading every minute, lossy radio', 'UDP')}
MINE = {'a': 'TCP', 'b': '', 'c': '', 'd': '', 'e': '', 'f': ''}   # ← your answers: TCP, UDP or UDP+RTP
for k, (app, sol) in P1.items():
    print(f"{'✅' if MINE[k].upper() == sol else '❌'} ({k}) {app:42s} → {sol}")

## B.2 Problems 2 and 3 — how long does a page take?
Base file 50 KB, 6 images of 100 KB, access link 20 Mbit/s (the bottleneck), RTT 40 ms.

In [ ]:
R, RTT = 20e6, 0.040
tb, ti = 50e3 * 8 / R, 100e3 * 8 / R
check('P2.1 base file transmission (ms)', tb * 1e3, 20); check('P2.1 one image (ms)', ti * 1e3, 40)
check('P2.2 non-persistent (ms)', page_load(6, RTT, tb, ti) * 1e3, 820)
check('P2.3 persistent (ms)', page_load(6, RTT, tb, ti, 'persistent') * 1e3, 580)
check('P2.4 pipelined (ms)', page_load(6, RTT, tb, ti, 'pipelined') * 1e3, 380)
t_small = 2e3 * 8 / R
print('P2.5 — 60 images of 2 KB each (after the base page):')
for mode in ('nonpersistent', 'persistent', 'pipelined'):
    print(f'   {mode:14s} {(page_load(60, RTT, tb, t_small, mode) - (2*RTT + tb)) * 1e3:7.0f} ms')
check('P3.1 six parallel connections sharing the link (ms)', page_load(6, RTT, tb, ti, parallel=6, shared_link=True) * 1e3, 420)
print('P3.2 parallelism hides round trips, not transmission: 6 × 40 ms still share one 20 Mbit/s link.')

## B.3 Problem 4 — reading an HTTP exchange
The exchange of the problem is parsed by code, and then **replayed** against the mini server of A.6, which answers `304 Not Modified` exactly as in the sheet.

In [ ]:
REQ = """GET /files/notes.pdf HTTP/1.1
Host: cn.example.edu
User-Agent: curl/8.5.0
If-Modified-Since: Fri, 16 Oct 2026 14:12:30 GMT
Accept: */*
"""
def parse_head(text):
    lines = text.strip().splitlines()
    return lines[0], {k.strip().lower(): v.strip() for k, _, v in (l.partition(':') for l in lines[1:])}

first, hd = parse_head(REQ)
method, target, version = first.split()
print('1. URL       : http://' + hd['host'] + target)
print('2. program   :', hd['user-agent'], '| persistent:', version == 'HTTP/1.1' and hd.get('connection') != 'close')
print('3. asks      : send the object only if modified after', hd['if-modified-since'])

s = socket.create_connection(('127.0.0.1', web.port)); rf = s.makefile('rb')
status, h, body = exchange(s, rf, 'GET', target, host=hd['host'],
                           headers={'User-Agent': hd['user-agent'], 'If-Modified-Since': hd['if-modified-since'], 'Accept': '*/*'})
print('4. body bytes:', len(body), '(a 304 never has a body; the header ends at the empty line)')
print('   and without If-Modified-Since:'); exchange(s, rf, 'GET', target, close=True)
rf.close(); s.close()      # close the file object too, or the connection stays open

## B.4 Problem 5 — chunked bodies

In [ ]:
raw = b'1a\r\nabcdefghijklmnopqrstuvwxyz\r\na\r\n0123456789\r\n0\r\n\r\n'
body, n = decode_chunked(raw)
print(body, f'→ {len(body)} bytes in {n} chunks (0x1a = 26, 0xa = 10)')
check('length', len(body), 36)

## B.5 Problem 6 — what the receiver sees
To show that **any split** of the byte stream is possible, we put a small relay between client and server that forwards the bytes in random-sized pieces, with short pauses. The client calls `send("HELLO")` and `send("WORLD")`; the server calls `recv(100)` repeatedly.

In [ ]:
def resegmenting_relay(target_port, seed):
    """Accepts one connection and forwards its bytes to target_port in random-sized pieces."""
    rng = random.Random(seed)
    ls_ = socket.socket(); ls_.bind(('127.0.0.1', 0)); ls_.listen(1)
    def run():
        a, _ = ls_.accept(); data = b''
        a.settimeout(0.3)
        try:
            while chunk := a.recv(100): data += chunk
        except socket.timeout:
            pass
        out = socket.create_connection(('127.0.0.1', target_port))
        out.setsockopt(socket.IPPROTO_TCP, socket.TCP_NODELAY, 1)
        while data:
            k = rng.randint(1, len(data)); out.sendall(data[:k]); data = data[k:]; time.sleep(0.05)
        out.close(); a.close(); ls_.close()
    threading.Thread(target=run, daemon=True).start()
    return ls_.getsockname()[1]

def receiver():
    ls_ = socket.socket(); ls_.bind(('127.0.0.1', 0)); ls_.listen(1)
    got = []
    def run():
        c, _ = ls_.accept()
        while d := c.recv(100): got.append(d)
        c.close(); ls_.close()
    t = threading.Thread(target=run, daemon=True); t.start()
    return ls_.getsockname()[1], got, t

for seed in (1, 2, 3, 4):
    port, got, t = receiver()
    relay = resegmenting_relay(port, seed)
    c = socket.create_connection(('127.0.0.1', relay)); c.send(b'HELLO'); c.send(b'WORLD'); c.close()
    t.join(5)
    print(f'run {seed}: recv returned {[len(g) for g in got]}  {got}')

In [ ]:
# Two ways to make the boundaries visible again
def frame_lines(msgs):  return b''.join(m + b'\n' for m in msgs)
def frame_length(msgs): return b''.join(struct.pack('!I', len(m)) + m for m in msgs)

def parse_lines(stream):
    *msgs, rest = stream.split(b'\n'); return msgs, rest

def parse_length(stream):
    msgs, pos = [], 0
    while pos + 4 <= len(stream):
        n = struct.unpack('!I', stream[pos:pos + 4])[0]
        if pos + 4 + n > len(stream): break
        msgs.append(stream[pos + 4:pos + 4 + n]); pos += 4 + n
    return msgs, stream[pos:]

for name, framer, parser in [('delimiter (newline)', frame_lines, parse_lines), ('length prefix (4 bytes, network order)', frame_length, parse_length)]:
    wire = framer([b'HELLO', b'WORLD'])
    print(f'{name}: {len(wire)} bytes on the wire: {wire.hex(" ")}')
    port, got, t = receiver(); relay = resegmenting_relay(port, 7)
    c = socket.create_connection(('127.0.0.1', relay)); c.sendall(wire); c.close(); t.join(5)
    msgs, rest = parser(b''.join(got))
    print(f'   arrived in pieces {[len(g) for g in got]} → messages recovered: {msgs}\n')

## B.6 Problems 7 and 8 — status codes and repeatable requests
Each situation of Problem 7 is sent to the mini server; compare its answer with yours.

In [ ]:
SITUATIONS = [('site moved for good to https', 'GET', '/old', {}),
              ('/admin without logging in', 'GET', '/admin', {}),
              ('/admin as a logged-in student', 'GET', '/admin', {'Authorization': 'student'}),
              ('DELETE /index.html', 'DELETE', '/index.html', {}),
              ('new account created by POST', 'POST', '/orders', {}),
              ('the application crashed', 'GET', '/crash', {}),
              ('request line "GTE / HTTP/1.1"', 'GTE', '/', {})]
EXPECTED = ['301', '401', '403', '405', '201', '500', '501']
for (what, m, tgt, hdrs), exp in zip(SITUATIONS, EXPECTED):
    with socket.create_connection(('127.0.0.1', web.port)) as s:
        s.sendall(request(m, tgt, headers=hdrs, body=b'x' if m == 'POST' else b'', close=True))
        st = s.makefile('rb').readline().decode().strip()
    print(f"{'✅' if st.split()[1] == exp else '❌'} {what:34s} → {st}")
print('(For GTE, 400 Bad Request is also acceptable.)')

In [ ]:
P8 = [('GET /timetable?group=CS2', True, 'safe and idempotent'),
      ('PUT /profile/photo', True, 'idempotent: same photo twice, same result'),
      ('DELETE /cart/items/17', True, 'idempotent: the second finds nothing (404 is fine)'),
      ('POST /orders', False, 'NOT idempotent: the order may already exist')]
for req_, ok, why in P8:
    print(f'{req_:28s} repeat automatically? {"yes" if ok else "NO "}  — {why}')

print('\nThe fix for POST: an idempotency key. The reply to the first POST is "lost"; the app retries:')
n0 = len(web.orders)
for attempt in (1, 2):
    with socket.create_connection(('127.0.0.1', web.port)) as s:
        s.sendall(request('POST', '/orders', headers={'Idempotency-Key': 'cart-7f3a'}, body=b'2 x notebook', close=True))
        st, _, bd, _ = read_response(s.makefile('rb'), 'POST')
    print(f'   attempt {attempt}: {st} {bd.decode().strip()}{"   (reply lost)" if attempt == 1 else ""}')
print(f'   orders actually created: {len(web.orders) - n0}')

## B.7 Problem 9 — sockets and 4-tuples on a busy server
Two connections from a "laptop" (127.0.0.1) and one from a "phone" (we bind its socket to 127.0.0.2, another loopback address).

In [ ]:
srv = socket.socket(); srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1); srv.bind(('127.0.0.1', 0)); srv.listen(8)
port = srv.getsockname()[1]
laptop = [socket.create_connection(('127.0.0.1', port)) for _ in range(2)]
phone = socket.socket(); phone.bind(('127.0.0.2', 0)); phone.connect(('127.0.0.1', port))
accepted = [srv.accept()[0] for _ in range(3)]
print(f'Server sockets: 1 listening + {len(accepted)} connected = {1 + len(accepted)}')
for c in accepted:
    print('   4-tuple', c.getpeername() + c.getsockname())
sh(f'ss -tn state established "( sport = :{port} )"')
for x in laptop + [phone] + accepted + [srv]: x.close()

## B.8 Problem 10 — design and implement a file-upload protocol
The seminar's answer, implemented: **request** `PUT <name> <size>\r\n` followed by exactly `<size>` raw bytes; **response** `OK\r\n` or `ERR <code> <text>\r\n`. The header is framed by a delimiter, the body by its length — the two rules of HTTP/1.1. The test file contains newlines and zero bytes on purpose.

In [ ]:
class UploadServer(threading.Thread):
    MAX = 10_000_000
    def __init__(self):
        super().__init__(daemon=True)
        self.ls = socket.socket(); self.ls.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
        self.ls.bind(('127.0.0.1', 0)); self.ls.listen(8); self.port = self.ls.getsockname()[1]
        self.files, self.log = {}, []
        self.start()
    def run(self):
        while True:
            try: c, _ = self.ls.accept()
            except OSError: return
            with c:
                rf = c.makefile('rb')
                while (line := rf.readline()):
                    if not line.endswith(b'\r\n'):
                        self.log.append('connection closed in the middle of a header line'); break
                    parts = line[:-2].decode('ascii', 'replace').split(' ')
                    if len(parts) != 3 or parts[0] != 'PUT' or not parts[2].isdigit():
                        c.sendall(b'ERR 400 bad request\r\n'); continue
                    name, size = parts[1], int(parts[2])
                    if size > self.MAX:
                        c.sendall(b'ERR 413 too large\r\n'); break
                    data = rf.read(size)
                    if len(data) < size:            # homework: the connection closed before <size> bytes
                        self.log.append(f'{name}: only {len(data)} of {size} bytes arrived → partial file discarded, no reply possible')
                        break
                    self.files[name] = data
                    c.sendall(b'OK\r\n')

def upload(sock, rf, name, data):
    sock.sendall(f'PUT {name} {len(data)}\r\n'.encode() + data)
    return rf.readline().decode().strip()

up = UploadServer()
payload = b'line 1\nline 2\r\n\x00\x00binary\x00\xff' * 1000
s = socket.create_connection(('127.0.0.1', up.port)); rf = s.makefile('rb')
print('upload 1 :', upload(s, rf, 'data.bin', payload))
print('upload 2 :', upload(s, rf, 'small.txt', b'hi\n'), ' (same connection)')
print('bad req  :', (s.sendall(b'GET data.bin\r\n'), rf.readline().decode().strip())[1])
print('repeat 1 :', upload(s, rf, 'data.bin', payload), ' (PUT is idempotent: same result)')
rf.close(); s.close(); time.sleep(0.2)
print('stored intact:', up.files['data.bin'] == payload, f'({len(payload)} bytes)')

# Homework: the client disappears in the middle of the body
s = socket.create_connection(('127.0.0.1', up.port))
s.sendall(b'PUT broken.bin 1000\r\n' + b'x' * 300); s.close(); time.sleep(0.3)
print('server log:', up.log[-1] if up.log else '(nothing logged yet)')

**Homework**: exercises 1–5 of the lecture (see A.8); compare `curl -v` and `curl -v --http1.0` (below); what should the Problem 10 server do if the connection closes early (see the last line above).

In [ ]:
for opt in ('', '--http1.0'):
    print(f'### curl -v {opt} http://example.org/')
    out = sh(f'curl -sv {opt} http://example.org/ -o /dev/null 2>&1 | grep -E "^[<>] |^\\* (Connection|Closing|shutting)" | head -20', show=False)
    print(out or '(not reachable)')

**✍️ Your answers** (double-click to edit)

- **Homework 2 — differences between the HTTP/1.1 and HTTP/1.0 requests, and the Connection behaviour:** 

---
# Part C · Laboratory 3 (Linux) — Sockets I: a TCP client and server in C
The `%%writefile` cells below create the files of the laboratory archive in the folder `lab3/`. Read each program before running it — the comments follow the numbered steps of Lecture 3, Figure 2. Port numbers: 5000–5004 as in the sheet.

### The files

In [ ]:
%%writefile lab3/netio.h
/* netio.h - helpers for TCP byte streams (Computer Networks, Lab 3) */
#ifndef NETIO_H
#define NETIO_H
#include <stdint.h>
#include <errno.h>
#include <unistd.h>
#include <arpa/inet.h>
#include <sys/socket.h>

/* Write exactly len bytes. Returns 0 on success, -1 on error. */
static inline int send_all(int fd, const void *buf, size_t len) {
    const char *p = buf;
    while (len > 0) {
        ssize_t n = send(fd, p, len, 0);
        if (n < 0) { if (errno == EINTR) continue; return -1; }
        p += n; len -= (size_t)n;
    }
    return 0;
}

/* Read exactly len bytes. Returns 1 on success, 0 if the peer closed first, -1 on error. */
static inline int recv_all(int fd, void *buf, size_t len) {
    char *p = buf;
    while (len > 0) {
        ssize_t n = recv(fd, p, len, 0);
        if (n < 0) { if (errno == EINTR) continue; return -1; }
        if (n == 0) return 0;
        p += n; len -= (size_t)n;
    }
    return 1;
}

/* Length-prefixed messages: 4-byte length in network byte order, then the payload. */
static inline int send_msg(int fd, const void *buf, uint32_t len) {
    uint32_t netlen = htonl(len);
    if (send_all(fd, &netlen, sizeof netlen) < 0) return -1;
    return send_all(fd, buf, len);
}

/* Returns the payload length, 0 if the peer closed, -1 on error or if the message exceeds cap. */
static inline long recv_msg(int fd, void *buf, uint32_t cap) {
    uint32_t netlen;
    int r = recv_all(fd, &netlen, sizeof netlen);
    if (r <= 0) return r;
    uint32_t len = ntohl(netlen);
    if (len > cap) return -1;
    r = recv_all(fd, buf, len);
    return r <= 0 ? -1 : (long)len;
}
#endif

In [ ]:
%%writefile lab3/echo_client.c
/* echo_client.c - send each line of standard input to a TCP server, print what comes back.
   Usage: ./echo_client HOST PORT */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <netdb.h>
#include <sys/socket.h>
#include "netio.h"

int main(int argc, char *argv[]) {
    if (argc != 3) { fprintf(stderr, "usage: %s HOST PORT\n", argv[0]); return 1; }
    setvbuf(stdout, NULL, _IOLBF, 0);
    struct addrinfo hints = {0}, *res, *ai;
    hints.ai_family = AF_UNSPEC;          /* IPv4 or IPv6 */
    hints.ai_socktype = SOCK_STREAM;      /* TCP */
    int err = getaddrinfo(argv[1], argv[2], &hints, &res);
    if (err) { fprintf(stderr, "getaddrinfo: %s\n", gai_strerror(err)); return 1; }

    int fd = -1;
    for (ai = res; ai != NULL; ai = ai->ai_next) {          /* try each address in turn */
        fd = socket(ai->ai_family, ai->ai_socktype, ai->ai_protocol);
        if (fd < 0) continue;
        if (connect(fd, ai->ai_addr, ai->ai_addrlen) == 0) break;
        close(fd); fd = -1;
    }
    freeaddrinfo(res);
    if (fd < 0) { perror("connect"); return 1; }

    char line[1024], reply[1024];
    while (fgets(line, sizeof line, stdin) != NULL) {
        size_t len = strlen(line);
        if (send_all(fd, line, len) < 0) { perror("send"); break; }
        /* The echo has the same length as what we sent, but may arrive in pieces. */
        int r = recv_all(fd, reply, len);
        if (r <= 0) { fprintf(stderr, r == 0 ? "server closed the connection\n" : "recv failed\n"); break; }
        fwrite(reply, 1, len, stdout);
    }
    close(fd);
    return 0;
}

In [ ]:
%%writefile lab3/echo_server.c
/* echo_server.c - iterative TCP echo server: one client at a time.
   Usage: ./echo_server PORT */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <arpa/inet.h>
#include <netinet/in.h>
#include <sys/socket.h>
#include "netio.h"

int main(int argc, char *argv[]) {
    if (argc != 2) { fprintf(stderr, "usage: %s PORT\n", argv[0]); return 1; }
    setvbuf(stdout, NULL, _IOLBF, 0);                         /* print each line at once */
    int lfd = socket(AF_INET, SOCK_STREAM, 0);               /* 1. create */
    if (lfd < 0) { perror("socket"); return 1; }
    int on = 1;
    setsockopt(lfd, SOL_SOCKET, SO_REUSEADDR, &on, sizeof on); /* allow quick restarts */
    struct sockaddr_in addr = {0};
    addr.sin_family = AF_INET;
    addr.sin_addr.s_addr = htonl(INADDR_ANY);                 /* all local addresses */
    addr.sin_port = htons((uint16_t)atoi(argv[1]));            /* host -> network byte order */
    if (bind(lfd, (struct sockaddr *)&addr, sizeof addr) < 0) { perror("bind"); return 1; } /* 2. bind */
    if (listen(lfd, 8) < 0) { perror("listen"); return 1; }  /* 3. listen, backlog 8 */
    printf("listening on port %s\n", argv[1]);

    for (;;) {
        struct sockaddr_in peer; socklen_t plen = sizeof peer;
        int cfd = accept(lfd, (struct sockaddr *)&peer, &plen); /* 4. accept: a NEW socket */
        if (cfd < 0) { perror("accept"); continue; }
        char ip[INET_ADDRSTRLEN];
        inet_ntop(AF_INET, &peer.sin_addr, ip, sizeof ip);
        printf("connection from %s:%u\n", ip, ntohs(peer.sin_port));
        char buf[65536]; ssize_t n; long total = 0;
        while ((n = recv(cfd, buf, sizeof buf, 0)) > 0) {      /* 5. read what has arrived */
            printf("  recv returned %zd bytes\n", n);
            total += n;
            if (send_all(cfd, buf, (size_t)n) < 0) { perror("send"); break; }
        }
        printf("client closed after %ld bytes\n", total);
        close(cfd);                                            /* 6. close this connection */
    }
}

In [ ]:
%%writefile lab3/upper_server.c
/* upper_server.c - line-based protocol: each line ending in \n comes back in upper case.
   Usage: ./upper_server PORT */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>
#include <unistd.h>
#include <netinet/in.h>
#include <sys/socket.h>
#include "netio.h"

int main(int argc, char *argv[]) {
    if (argc != 2) { fprintf(stderr, "usage: %s PORT\n", argv[0]); return 1; }
    setvbuf(stdout, NULL, _IOLBF, 0);
    int lfd = socket(AF_INET, SOCK_STREAM, 0);
    if (lfd < 0) { perror("socket"); return 1; }
    int on = 1;
    setsockopt(lfd, SOL_SOCKET, SO_REUSEADDR, &on, sizeof on);
    struct sockaddr_in addr = {0};
    addr.sin_family = AF_INET;
    addr.sin_addr.s_addr = htonl(INADDR_ANY);
    addr.sin_port = htons((uint16_t)atoi(argv[1]));
    if (bind(lfd, (struct sockaddr *)&addr, sizeof addr) < 0) { perror("bind"); return 1; }
    if (listen(lfd, 8) < 0) { perror("listen"); return 1; }
    printf("listening on port %s\n", argv[1]);

    for (;;) {
        int cfd = accept(lfd, NULL, NULL);
        if (cfd < 0) continue;
        char buf[4096]; size_t used = 0; ssize_t n;
        while ((n = recv(cfd, buf + used, sizeof buf - used, 0)) > 0) {
            used += (size_t)n;
            char *nl;
            /* A single recv may hold part of a line, one line, or several lines. */
            while ((nl = memchr(buf, '\n', used)) != NULL) {
                size_t linelen = (size_t)(nl - buf) + 1;
                for (size_t i = 0; i < linelen; i++) buf[i] = (char)toupper((unsigned char)buf[i]);
                if (send_all(cfd, buf, linelen) < 0) break;
                memmove(buf, buf + linelen, used - linelen);   /* keep the rest for later */
                used -= linelen;
            }
            if (used == sizeof buf) { const char *e = "ERR line too long\n"; send_all(cfd, e, strlen(e)); break; }
        }
        if (used > 0) printf("connection closed with %zu bytes of an incomplete line discarded\n", used);
        close(cfd);
    }
}

In [ ]:
%%writefile lab3/blast.c
/* blast.c - send N bytes with ONE call to send_all, close the sending direction, then read the echo.
   Usage: ./blast HOST PORT N */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <netdb.h>
#include <sys/socket.h>
#include "netio.h"

int main(int argc, char *argv[]) {
    if (argc != 4) { fprintf(stderr, "usage: %s HOST PORT N\n", argv[0]); return 1; }
    size_t n = (size_t)strtoul(argv[3], NULL, 10);
    struct addrinfo hints = {0}, *res;
    hints.ai_family = AF_UNSPEC; hints.ai_socktype = SOCK_STREAM;
    int err = getaddrinfo(argv[1], argv[2], &hints, &res);
    if (err) { fprintf(stderr, "getaddrinfo: %s\n", gai_strerror(err)); return 1; }
    int fd = socket(res->ai_family, res->ai_socktype, res->ai_protocol);
    if (fd < 0 || connect(fd, res->ai_addr, res->ai_addrlen) < 0) { perror("connect"); return 1; }
    freeaddrinfo(res);

    char *data = malloc(n ? n : 1);
    if (!data) { perror("malloc"); return 1; }
    memset(data, 'x', n);
    if (send_all(fd, data, n) < 0) { perror("send"); return 1; }   /* ONE call for all N bytes */
    shutdown(fd, SHUT_WR);                                          /* "I have finished sending" */

    char buf[65536]; ssize_t r; size_t back = 0;                    /* drain the echo */
    while ((r = recv(fd, buf, sizeof buf, 0)) > 0) back += (size_t)r;
    printf("sent %zu bytes in one send_all, received %zu bytes back\n", n, back);
    free(data); close(fd);
    return 0;
}

In [ ]:
%%writefile lab3/msg_server.c
/* msg_server.c - length-prefixed messages: replies "OK <n>" (as a message) to every message of n bytes.
   Usage: ./msg_server PORT */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <netinet/in.h>
#include <sys/socket.h>
#include "netio.h"

#define CAP (1u << 20)                                   /* largest message accepted: 1 MiB */

int main(int argc, char *argv[]) {
    if (argc != 2) { fprintf(stderr, "usage: %s PORT\n", argv[0]); return 1; }
    setvbuf(stdout, NULL, _IOLBF, 0);
    int lfd = socket(AF_INET, SOCK_STREAM, 0);
    if (lfd < 0) { perror("socket"); return 1; }
    int on = 1;
    setsockopt(lfd, SOL_SOCKET, SO_REUSEADDR, &on, sizeof on);
    struct sockaddr_in addr = {0};
    addr.sin_family = AF_INET;
    addr.sin_addr.s_addr = htonl(INADDR_ANY);
    addr.sin_port = htons((uint16_t)atoi(argv[1]));
    if (bind(lfd, (struct sockaddr *)&addr, sizeof addr) < 0) { perror("bind"); return 1; }
    if (listen(lfd, 8) < 0) { perror("listen"); return 1; }
    printf("listening on port %s\n", argv[1]);

    char *buf = malloc(CAP);
    if (!buf) { perror("malloc"); return 1; }
    for (;;) {
        int cfd = accept(lfd, NULL, NULL);
        if (cfd < 0) continue;
        long len;
        while ((len = recv_msg(cfd, buf, CAP)) > 0) {
            printf("message of %ld bytes\n", len);
            char reply[32];
            int rl = snprintf(reply, sizeof reply, "OK %ld", len);
            if (send_msg(cfd, reply, (uint32_t)rl) < 0) break;
        }
        if (len < 0) printf("bad or too long message (limit %u bytes): closing\n", CAP);
        else printf("client closed\n");
        close(cfd);
    }
}

In [ ]:
%%writefile lab3/msg_client.c
/* msg_client.c - send each argument as a length-prefixed message, ALL before reading any reply.
   Usage: ./msg_client HOST PORT MSG... */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <unistd.h>
#include <netdb.h>
#include <sys/socket.h>
#include "netio.h"

int main(int argc, char *argv[]) {
    if (argc < 4) { fprintf(stderr, "usage: %s HOST PORT MSG...\n", argv[0]); return 1; }
    struct addrinfo hints = {0}, *res;
    hints.ai_family = AF_UNSPEC; hints.ai_socktype = SOCK_STREAM;
    int err = getaddrinfo(argv[1], argv[2], &hints, &res);
    if (err) { fprintf(stderr, "getaddrinfo: %s\n", gai_strerror(err)); return 1; }
    int fd = socket(res->ai_family, res->ai_socktype, res->ai_protocol);
    if (fd < 0 || connect(fd, res->ai_addr, res->ai_addrlen) < 0) { perror("connect"); return 1; }
    freeaddrinfo(res);

    for (int i = 3; i < argc; i++)                                   /* send them all first */
        if (send_msg(fd, argv[i], (uint32_t)strlen(argv[i])) < 0) { perror("send"); return 1; }
    for (int i = 3; i < argc; i++) {                                 /* then read the replies */
        char reply[64];
        long n = recv_msg(fd, reply, sizeof reply - 1);
        if (n <= 0) { fprintf(stderr, "no reply\n"); break; }
        reply[n] = '\0';
        printf("%s -> %s\n", argv[i], reply);
    }
    close(fd);
    return 0;
}

In [ ]:
%%writefile lab3/file_client.c
/* file_client.c - send a whole file as ONE length-prefixed message and print the server's reply.
   Usage: ./file_client HOST PORT FILE */
#include <stdio.h>
#include <stdlib.h>
#include <unistd.h>
#include <netdb.h>
#include <sys/socket.h>
#include <sys/stat.h>
#include "netio.h"

int main(int argc, char *argv[]) {
    if (argc != 4) { fprintf(stderr, "usage: %s HOST PORT FILE\n", argv[0]); return 1; }
    FILE *f = fopen(argv[3], "rb");
    if (!f) { perror("fopen"); return 1; }
    struct stat st;
    if (fstat(fileno(f), &st) < 0) { perror("stat"); return 1; }
    size_t size = (size_t)st.st_size;
    char *data = malloc(size ? size : 1);
    if (!data || fread(data, 1, size, f) != size) { fprintf(stderr, "cannot read file\n"); return 1; }
    fclose(f);

    struct addrinfo hints = {0}, *res;
    hints.ai_family = AF_UNSPEC; hints.ai_socktype = SOCK_STREAM;
    int err = getaddrinfo(argv[1], argv[2], &hints, &res);
    if (err) { fprintf(stderr, "getaddrinfo: %s\n", gai_strerror(err)); return 1; }
    int fd = socket(res->ai_family, res->ai_socktype, res->ai_protocol);
    if (fd < 0 || connect(fd, res->ai_addr, res->ai_addrlen) < 0) { perror("connect"); return 1; }
    freeaddrinfo(res);

    if (send_msg(fd, data, (uint32_t)size) < 0) perror("send");
    char reply[64];
    long n = recv_msg(fd, reply, sizeof reply - 1);
    if (n > 0) { reply[n] = '\0'; printf("file of %zu bytes -> server says: %s\n", size, reply); }
    else printf("file of %zu bytes -> no reply (the server closed the connection)\n", size);
    free(data); close(fd);
    return 0;
}

In [ ]:
%%writefile lab3/broken_pipe.c
/* broken_pipe.c - keep writing to a connection after the server has closed it.
   Usage: ./broken_pipe HOST PORT [ignore]   ("ignore" sets SIGPIPE to SIG_IGN) */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <signal.h>
#include <unistd.h>
#include <netdb.h>
#include <sys/socket.h>

int main(int argc, char *argv[]) {
    if (argc < 3) { fprintf(stderr, "usage: %s HOST PORT [ignore]\n", argv[0]); return 1; }
    setvbuf(stdout, NULL, _IONBF, 0);
    if (argc == 4 && strcmp(argv[3], "ignore") == 0) signal(SIGPIPE, SIG_IGN);
    struct addrinfo hints = {0}, *res;
    hints.ai_family = AF_UNSPEC; hints.ai_socktype = SOCK_STREAM;
    if (getaddrinfo(argv[1], argv[2], &hints, &res)) { fprintf(stderr, "getaddrinfo failed\n"); return 1; }
    int fd = socket(res->ai_family, res->ai_socktype, res->ai_protocol);
    if (fd < 0 || connect(fd, res->ai_addr, res->ai_addrlen) < 0) { perror("connect"); return 1; }
    freeaddrinfo(res);
    for (int i = 1; i <= 5; i++) {
        sleep(1);                                    /* meanwhile the server closes the connection */
        ssize_t n = send(fd, "data\n", 5, 0);
        if (n < 0) { perror("send"); return 1; }
        printf("send %d returned %zd\n", i, n);
    }
    close(fd);
    return 0;
}

In [ ]:
%%writefile lab3/Makefile
CC = gcc
CFLAGS = -Wall -Wextra -O2
PROGS = echo_client echo_server upper_server blast msg_server msg_client file_client broken_pipe

all: $(PROGS)

%: %.c netio.h
	$(CC) $(CFLAGS) -o $@ $<

clean:
	rm -f $(PROGS)

In [ ]:
sh(f'cd {LAB} && make clean > /dev/null; make')
sh(f'ls -l {LAB}')

## Task 1 — A server without writing code
`nc -l 5000` listens; `nc localhost 5000` connects. The notebook plays both terminals.

In [ ]:
Bg.stop_all(); sh('pkill -f "nc -l 5000"', show=False)
A = Bg('nc -l 5000')                              # terminal A
sh('ss -tln | grep -E "State|:5000"')             # terminal C, before the client
B = Bg('nc -N localhost 5000')                    # terminal B   (-N: send FIN when stdin ends)
sh('ss -tn | grep -E "State|:5000"')              # terminal C, after
B.write('hello from B\n'); A.write('hello from A\n'); time.sleep(0.5)
show('terminal A (server) shows', A.output()); show('terminal B (client) shows', B.output())

In [ ]:
B.eof(); time.sleep(1)                            # Ctrl+D in terminal B
print('client still running:', B.running(), '| server still running:', A.running())
sh('ss -tan | grep -E "State|:5000"')
A.stop(); B.stop()

**✍️ Your answers** (double-click to edit)

- **Q1 — ss -tln before, ss -tn after; the 4-tuple; who chose the client port?:** 
- **Q2 — what happens to the server after Ctrl+D; which TCP message told it?:** 

## Task 2 — The echo client (first against `nc`, then without a server)

In [ ]:
A = Bg('nc -l 5000')
C = Bg('./echo_client localhost 5000')
C.write('ping\n'); time.sleep(0.4)
show('nc (server) received', A.output())
A.write('ping\n'); time.sleep(0.4)                # we play the echo by hand
show('echo_client printed', C.output())
C.eof(); A.stop(); C.stop()

In [ ]:
# Q4 — nothing listens on port 5000
sh(f'cd {LAB} && echo test | ./echo_client localhost 5000; echo "exit code $?"')

**✍️ Your answers** (double-click to edit)

- **Q3 — why recv_all(fd, reply, len) and not one recv?:** 
- **Q4 — the error and the TCP message that caused it:** 

## Task 3 — The echo server (iterative: one client at a time)

In [ ]:
S = Bg('./echo_server 5000')
B = Bg('./echo_client localhost 5000'); B.write('first client, line 1\n')
C = Bg('./echo_client localhost 5000'); C.write('second client waits\n')
time.sleep(1)
show('client B', B.output()); show('client C (no echo yet!)', C.output())
sh('ss -tn | grep -E "State|:5000"')              # is C's connection established?
B.eof(); time.sleep(1)                            # B quits
show('client C after B quits', C.output())
C.eof(); time.sleep(0.5)
show('server log', S.output())

In [ ]:
# Q7 — remove htons from the line that sets sin_port
sh(f"cd {LAB} && sed 's/htons((uint16_t)atoi(argv\\[1\\]))/(uint16_t)atoi(argv[1])/' echo_server.c > echo_server_nohtons.c "
   f"&& grep -n 'sin_port' echo_server_nohtons.c && gcc -Wall -O2 -o echo_server_nohtons echo_server_nohtons.c")
S.stop()
N = Bg('./echo_server_nohtons 5000')
show('it says', N.output())
sh('ss -tlnp | grep -E "State|echo_server"')          # look at the Local Address:Port column
print('Explanation: 5000 =', f'0x{5000:04X}', '→ stored as bytes', struct.pack('<H', 5000).hex(' '),
      '→ read in network order as', struct.unpack('!H', struct.pack('<H', 5000))[0])
N.stop()

**✍️ Your answers** (double-click to edit)

- **Q5 — the server steps vs Figure 2; what it prints; where the client port comes from:** 
- **Q6 — why the second client waits; is its connection established?:** 
- **Q7 — the real port without htons, explained:** 

## Task 4 — Sockets as the operating system sees them

In [ ]:
S = Bg('./echo_server 5000'); B = Bg('./echo_client localhost 5000'); B.write('x\n'); time.sleep(0.4)
sh('ss -tnp | grep -E "5000|echo"')
sh(f'ls -l /proc/{S.pid}/fd')
B.eof(); B.stop()

**✍️ Your answers** (double-click to edit)

- **Q8 — how many sockets echo_server holds with one client, in which states, which descriptors:** 

## Task 5 — A stream is not a sequence of messages
`blast` sends N bytes with **one** `send_all` and closes its sending direction. How many `recv` calls does the server need?

In [ ]:
S.new_output()
def recv_sizes(log):
    return [int(x) for x in re.findall(r'recv returned (\d+)', log)]

results = []
for n in (10, 5000, 300000, 300000, 300000, 3000000):
    out = sh(f'cd {LAB} && ./blast localhost 5000 {n}', show=False).strip()
    time.sleep(0.3)
    sizes = recv_sizes(S.new_output())
    results.append((n, sizes))
    print(f'N = {n:8d}: {len(sizes):3d} recv calls  sizes {sizes[:8]}{" ..." if len(sizes) > 8 else ""}')
print('\n', out)

In [ ]:
big = [s for n, sz in results if n == 3000000 for s in sz]
plt.hist(big, bins=30); plt.xlabel('bytes returned by one recv()'); plt.ylabel('count')
plt.title('One send_all of 3 000 000 bytes, as the server received it'); plt.show()
print('Largest possible recv here:', 'sizeof buf = 65536 bytes; also limited by what has arrived in the socket buffer')
sh('sysctl net.ipv4.tcp_rmem net.ipv4.tcp_wmem')

In [ ]:
# Q10 — the server's buffer reduced to 7 bytes
S.stop()
sh(f"cd {LAB} && sed 's/char buf\\[65536\\]/char buf[7]/' echo_server.c > echo_server7.c && gcc -Wall -O2 -o echo_server7 echo_server7.c")
S7 = Bg('./echo_server7 5000')
B = Bg('./echo_client localhost 5000'); B.write('hello world\n'); time.sleep(0.5)
show('server with a 7-byte buffer', S7.output()); show('client', B.output())
B.eof(); S7.stop(); B.stop()

**✍️ Your answers** (double-click to edit)

- **Q9 — one send, how many receives? Same sizes each time? What limits one recv?:** 
- **Q10 — server output with a 7-byte buffer; why the client still gets the full echo:** 

## Task 6 — Framing with lines (`upper_server`)

In [ ]:
U = Bg('./upper_server 5001')
sh(f"printf 'abc\\ndef\\nghi' | nc -N localhost 5001")
time.sleep(0.3); show('server log', U.output())

In [ ]:
# Q12 — a line longer than the server's 4096-byte buffer
reply = b''
s = socket.create_connection(('127.0.0.1', 5001))
try:
    s.sendall(b'a' * 5000 + b'\n')
    time.sleep(0.3)
    while d := s.recv(65536): reply += d
except OSError as e:
    print(f'{type(e).__name__}: the server closed while some of our bytes were still unread, so its TCP answered with RST')
finally:
    s.close()
print('received:', reply if reply else '(nothing — the RST can destroy the ERR line too)')
U.stop()

**✍️ Your answers** (double-click to edit)

- **Q11 — why "ghi" is never answered; what a protocol should say about an incomplete last line:** 
- **Q12 — behaviour with a line longer than 4096 bytes; is it good; a better design:** 

## Task 7 — Framing with a length prefix

In [ ]:
MS = Bg('./msg_server 5002')
sh(f'cd {LAB} && ./msg_client localhost 5002 short "a longer message" x')
wire = b''.join(struct.pack('!I', len(m)) + m for m in (b'short', b'a longer message', b'x'))
print(f'The client sent {len(wire)} bytes:'); print(' ', wire.hex(' '))

In [ ]:
# Q14 — a whole file as one message (file_client.c is a model solution)
sh(f'cd {LAB} && head -c 200000 /dev/urandom > small.bin && head -c 2000000 /dev/urandom > big.bin && stat -c "%n %s bytes" small.bin big.bin')
sh(f'cd {LAB} && ./file_client localhost 5002 small.bin')
sh(f'cd {LAB} && ./file_client localhost 5002 big.bin')
time.sleep(0.3); show('msg_server log', MS.output()); MS.stop()

**✍️ Your answers** (double-click to edit)

- **Q13 — how the server finds each message end; why htonl:** 
- **Q14 — file size vs reported size; what happens above 1 MB, and why:** 

## Task 8 — Breaking things on purpose

In [ ]:
print('1) Connection refused')
sh(f'cd {LAB} && echo x | ./echo_client localhost 5999')

print('2) Address already in use (server without SO_REUSEADDR, restarted at once)')
sh(f"cd {LAB} && grep -v -E 'SO_REUSEADDR|int on = 1' echo_server.c > echo_server_noreuse.c && gcc -Wall -O2 -o echo_server_noreuse echo_server_noreuse.c")
S = Bg('./echo_server_noreuse 5003'); B = Bg('./echo_client localhost 5003'); B.write('x\n'); time.sleep(0.3)
S.stop(signal.SIGINT); B.eof(); time.sleep(0.5)          # Ctrl+C on the server while a client is connected
sh('ss -tan | grep -E "State|:5003"')
S2 = Bg('./echo_server_noreuse 5003'); show('restart', S2.output()); S2.stop(); B.stop()

In [ ]:
print('3) Peer closed: kill the client while the server waits in recv')
S = Bg('./echo_server 5000'); B = Bg('./echo_client localhost 5000'); B.write('abc\n'); time.sleep(0.3)
B.stop(signal.SIGINT); time.sleep(0.4); show('server', S.output())

print('\n4) Broken pipe: write after the server has closed the connection')
closer = socket.socket(); closer.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1); closer.bind(('127.0.0.1', 5004)); closer.listen(4)
def accept_and_close():
    for _ in range(2):
        c, _ = closer.accept(); c.close()                  # the server closes at once
threading.Thread(target=accept_and_close, daemon=True).start()
r = subprocess.run(f'cd {LAB} && ./broken_pipe localhost 5004', shell=True, capture_output=True, text=True)
print(r.stdout + r.stderr, f'→ return code {r.returncode}' + ('  (killed by signal 13 = SIGPIPE)' if r.returncode in (-13, 141) else ''))
r = subprocess.run(f'cd {LAB} && ./broken_pipe localhost 5004 ignore', shell=True, capture_output=True, text=True)
print(r.stdout + r.stderr, f'→ with SIGPIPE ignored: return code {r.returncode}')
closer.close(); Bg.stop_all()

**✍️ Your answers** (double-click to edit)

- **Q15 — the four error texts and the TCP mechanism behind each:** 
- **Q16 — why SO_REUSEADDR in development; is it safe in production?:** 

## The same programs in Python
`sendall` is `send_all`; the stream behaviour — and the need for `recv_all` — is identical.

In [ ]:
%%writefile lab3/python/echo_server.py
"""echo_server.py - iterative TCP echo server (Linux and Windows). Usage: python echo_server.py PORT"""
import socket, sys
port = int(sys.argv[1])
with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as ls:
    ls.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    ls.bind(("127.0.0.1", port))
    ls.listen(8)
    print(f"listening on 127.0.0.1:{port}", flush=True)
    while True:
        conn, (ip, cport) = ls.accept()          # a NEW socket for this client
        print(f"connection from {ip}:{cport}", flush=True)
        with conn:
            total = 0
            while data := conn.recv(65536):      # b"" means the client closed
                print(f"  recv returned {len(data)} bytes", flush=True)
                total += len(data)
                conn.sendall(data)               # sendall = send_all in C
            print(f"client closed after {total} bytes", flush=True)

In [ ]:
%%writefile lab3/python/echo_client.py
"""echo_client.py - send lines from standard input, print the echo. Usage: python echo_client.py HOST PORT"""
import socket, sys
def recv_all(sock, n):
    """Read exactly n bytes, or raise if the peer closes first."""
    chunks = []
    while n > 0:
        data = sock.recv(n)
        if not data:
            raise ConnectionError("server closed the connection")
        chunks.append(data)
        n -= len(data)
    return b"".join(chunks)
with socket.create_connection((sys.argv[1], int(sys.argv[2]))) as s:
    for line in sys.stdin:
        data = line.encode()
        s.sendall(data)
        print(recv_all(s, len(data)).decode(), end="")

In [ ]:
P = Bg(f'{sys.executable} python/echo_server.py 5000')
sh(f'cd {LAB} && printf "one\\ntwo\\nthree\\n" | {sys.executable} python/echo_client.py 127.0.0.1 5000')
sh(f'cd {LAB} && ./blast 127.0.0.1 5000 300000')       # the C client against the Python server
time.sleep(0.3); show('Python server', P.output()); P.stop()

---
# Part D · Laboratory 3 (Windows) — Sockets I with Winsock
Colab runs Linux, but it can **build Windows programs**: MinGW-w64 is a cross-compiler that produces real `.exe` files. Below, the Winsock sources of the sheet (and the ports asked for in Tasks 6–7) are compiled with `-Wall -Wextra`, and you can download the `.exe` files and run them on your Windows PC exactly as the sheet describes.

## What is different in Winsock

In [ ]:
DIFF = [('Initialisation', 'none', 'WSAStartup(MAKEWORD(2,2), &wsa) first; WSACleanup() at the end'),
        ('Headers and library', '<sys/socket.h> <netinet/in.h> <arpa/inet.h> <netdb.h>', '<winsock2.h> <ws2tcpip.h>; link ws2_32'),
        ('Socket type', 'int (a file descriptor)', 'SOCKET (an unsigned handle)'),
        ('Failure value', '-1', 'INVALID_SOCKET (socket, accept) or SOCKET_ERROR'),
        ('Closing', 'close(fd)', 'closesocket(s) — close does not work on sockets'),
        ('Error code', 'errno, perror', 'WSAGetLastError()'),
        ('Reading and writing', 'read/write or recv/send', 'only recv/send; lengths are int'),
        ('Broken pipe', 'SIGPIPE signal', 'no signal; send fails with 10053/10054')]
for what, lin, win in DIFF:
    print(f'{what:20s} | Linux: {lin:52s} | Windows: {win}')

print('\nCommon error numbers:')
for name, num, posix in [('WSAECONNREFUSED', 10061, 'ECONNREFUSED'), ('WSAEADDRINUSE', 10048, 'EADDRINUSE'),
                          ('WSAECONNRESET', 10054, 'ECONNRESET'), ('WSAECONNABORTED', 10053, 'ECONNABORTED'),
                          ('WSANOTINITIALISED', 10093, '(no equivalent: WSAStartup was not called)')]:
    print(f'   {num}  {name:18s} ↔ Linux {posix}')

### The Winsock sources

In [ ]:
%%writefile lab3/win/netio_win.h
/* netio_win.h - helpers for TCP byte streams on Windows (Winsock 2) - Computer Networks, Lab 3 */
#ifndef NETIO_WIN_H
#define NETIO_WIN_H
#include <winsock2.h>
#include <ws2tcpip.h>
#include <stdint.h>

/* Write exactly len bytes. Returns 0 on success, -1 on error. */
static inline int send_all(SOCKET s, const void *buf, size_t len) {
    const char *p = buf;
    while (len > 0) {
        int n = send(s, p, (int)len, 0);
        if (n == SOCKET_ERROR) return -1;
        p += n; len -= (size_t)n;
    }
    return 0;
}

/* Read exactly len bytes. Returns 1 on success, 0 if the peer closed first, -1 on error. */
static inline int recv_all(SOCKET s, void *buf, size_t len) {
    char *p = buf;
    while (len > 0) {
        int n = recv(s, p, (int)len, 0);
        if (n == SOCKET_ERROR) return -1;
        if (n == 0) return 0;
        p += n; len -= (size_t)n;
    }
    return 1;
}

static inline int send_msg(SOCKET s, const void *buf, uint32_t len) {
    uint32_t netlen = htonl(len);
    if (send_all(s, &netlen, sizeof netlen) < 0) return -1;
    return send_all(s, buf, len);
}

static inline long recv_msg(SOCKET s, void *buf, uint32_t cap) {
    uint32_t netlen;
    int r = recv_all(s, &netlen, sizeof netlen);
    if (r <= 0) return r;
    uint32_t len = ntohl(netlen);
    if (len > cap) return -1;
    r = recv_all(s, buf, len);
    return r <= 0 ? -1 : (long)len;
}
#endif

In [ ]:
%%writefile lab3/win/echo_client_win.c
/* echo_client_win.c - Winsock version of echo_client.c.
   Build:  gcc echo_client_win.c -o echo_client.exe -lws2_32     (MinGW-w64)
      or:  cl echo_client_win.c ws2_32.lib                         (Visual Studio)
   Usage:  echo_client.exe HOST PORT */
#include <stdio.h>
#include <string.h>
#include "netio_win.h"
#ifdef _MSC_VER
#pragma comment(lib, "ws2_32.lib")
#endif

int main(int argc, char *argv[]) {
    if (argc != 3) { fprintf(stderr, "usage: %s HOST PORT\n", argv[0]); return 1; }
    WSADATA wsa;
    if (WSAStartup(MAKEWORD(2, 2), &wsa) != 0) { fprintf(stderr, "WSAStartup failed\n"); return 1; }
    struct addrinfo hints = {0}, *res, *ai;
    hints.ai_family = AF_UNSPEC; hints.ai_socktype = SOCK_STREAM;
    int err = getaddrinfo(argv[1], argv[2], &hints, &res);
    if (err) { fprintf(stderr, "getaddrinfo failed: %d\n", err); WSACleanup(); return 1; }
    SOCKET s = INVALID_SOCKET;
    for (ai = res; ai != NULL; ai = ai->ai_next) {
        s = socket(ai->ai_family, ai->ai_socktype, ai->ai_protocol);
        if (s == INVALID_SOCKET) continue;
        if (connect(s, ai->ai_addr, (int)ai->ai_addrlen) == 0) break;
        closesocket(s); s = INVALID_SOCKET;
    }
    freeaddrinfo(res);
    if (s == INVALID_SOCKET) { fprintf(stderr, "connect failed: error %d\n", WSAGetLastError()); WSACleanup(); return 1; }
    char line[1024], reply[1024];
    while (fgets(line, sizeof line, stdin) != NULL) {
        size_t len = strlen(line);
        if (send_all(s, line, len) < 0) { fprintf(stderr, "send failed: %d\n", WSAGetLastError()); break; }
        int r = recv_all(s, reply, len);
        if (r <= 0) { fprintf(stderr, r == 0 ? "server closed the connection\n" : "recv failed\n"); break; }
        fwrite(reply, 1, len, stdout);
    }
    closesocket(s);
    WSACleanup();
    return 0;
}

In [ ]:
%%writefile lab3/win/echo_server_win.c
/* echo_server_win.c - Winsock version of echo_server.c (iterative, one client at a time).
   Build:  gcc echo_server_win.c -o echo_server.exe -lws2_32
   Usage:  echo_server.exe PORT */
#include <stdio.h>
#include <stdlib.h>
#include "netio_win.h"
#ifdef _MSC_VER
#pragma comment(lib, "ws2_32.lib")
#endif

int main(int argc, char *argv[]) {
    if (argc != 2) { fprintf(stderr, "usage: %s PORT\n", argv[0]); return 1; }
    setvbuf(stdout, NULL, _IONBF, 0);
    WSADATA wsa;
    if (WSAStartup(MAKEWORD(2, 2), &wsa) != 0) { fprintf(stderr, "WSAStartup failed\n"); return 1; }
    SOCKET ls = socket(AF_INET, SOCK_STREAM, IPPROTO_TCP);
    if (ls == INVALID_SOCKET) { fprintf(stderr, "socket: %d\n", WSAGetLastError()); return 1; }
    struct sockaddr_in addr = {0};
    addr.sin_family = AF_INET;
    /* Loopback only: avoids the Windows Firewall prompt. Use INADDR_ANY to accept other machines. */
    addr.sin_addr.s_addr = htonl(INADDR_LOOPBACK);
    addr.sin_port = htons((u_short)atoi(argv[1]));
    if (bind(ls, (struct sockaddr *)&addr, sizeof addr) == SOCKET_ERROR) { fprintf(stderr, "bind: %d\n", WSAGetLastError()); return 1; }
    if (listen(ls, 8) == SOCKET_ERROR) { fprintf(stderr, "listen: %d\n", WSAGetLastError()); return 1; }
    printf("listening on 127.0.0.1:%s\n", argv[1]);
    for (;;) {
        struct sockaddr_in peer; int plen = sizeof peer;
        SOCKET cs = accept(ls, (struct sockaddr *)&peer, &plen);
        if (cs == INVALID_SOCKET) { fprintf(stderr, "accept: %d\n", WSAGetLastError()); continue; }
        char ip[INET_ADDRSTRLEN];
        inet_ntop(AF_INET, &peer.sin_addr, ip, sizeof ip);
        printf("connection from %s:%u\n", ip, ntohs(peer.sin_port));
        char buf[65536]; int n; long total = 0;
        while ((n = recv(cs, buf, sizeof buf, 0)) > 0) {
            printf("  recv returned %d bytes\n", n);
            total += n;
            if (send_all(cs, buf, (size_t)n) < 0) break;
        }
        printf("client closed after %ld bytes\n", total);
        closesocket(cs);
    }
}

In [ ]:
%%writefile lab3/win/upper_server_win.c
/* upper_server_win.c - Winsock port of upper_server.c (Windows sheet, Task 6).
   Build:  gcc upper_server_win.c -o upper_server.exe -lws2_32
   Usage:  upper_server.exe PORT */
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <ctype.h>
#include "netio_win.h"
#ifdef _MSC_VER
#pragma comment(lib, "ws2_32.lib")
#endif

int main(int argc, char *argv[]) {
    if (argc != 2) { fprintf(stderr, "usage: %s PORT\n", argv[0]); return 1; }
    setvbuf(stdout, NULL, _IONBF, 0);
    WSADATA wsa;
    if (WSAStartup(MAKEWORD(2, 2), &wsa) != 0) { fprintf(stderr, "WSAStartup failed\n"); return 1; }
    SOCKET ls = socket(AF_INET, SOCK_STREAM, IPPROTO_TCP);                 /* SOCKET, not int */
    if (ls == INVALID_SOCKET) { fprintf(stderr, "socket: %d\n", WSAGetLastError()); return 1; }
    struct sockaddr_in addr = {0};
    addr.sin_family = AF_INET;
    addr.sin_addr.s_addr = htonl(INADDR_LOOPBACK);
    addr.sin_port = htons((u_short)atoi(argv[1]));
    if (bind(ls, (struct sockaddr *)&addr, sizeof addr) == SOCKET_ERROR) { fprintf(stderr, "bind: %d\n", WSAGetLastError()); return 1; }
    if (listen(ls, 8) == SOCKET_ERROR) { fprintf(stderr, "listen: %d\n", WSAGetLastError()); return 1; }
    printf("listening on 127.0.0.1:%s\n", argv[1]);
    for (;;) {
        SOCKET cs = accept(ls, NULL, NULL);
        if (cs == INVALID_SOCKET) continue;
        char buf[4096]; size_t used = 0; int n;
        while ((n = recv(cs, buf + used, (int)(sizeof buf - used), 0)) > 0) {   /* lengths are int */
            used += (size_t)n;
            char *nl;
            while ((nl = memchr(buf, '\n', used)) != NULL) {
                size_t linelen = (size_t)(nl - buf) + 1;
                for (size_t i = 0; i < linelen; i++) buf[i] = (char)toupper((unsigned char)buf[i]);
                if (send_all(cs, buf, linelen) < 0) break;
                memmove(buf, buf + linelen, used - linelen);
                used -= linelen;
            }
            if (used == sizeof buf) { const char *e = "ERR line too long\n"; send_all(cs, e, strlen(e)); break; }
        }
        closesocket(cs);                                                    /* closesocket, not close */
    }
}

In [ ]:
%%writefile lab3/win/msg_server_win.c
/* msg_server_win.c - Winsock port of msg_server.c (Windows sheet, Task 7): replies "OK <n>".
   Build:  gcc msg_server_win.c -o msg_server.exe -lws2_32
   Usage:  msg_server.exe PORT */
#include <stdio.h>
#include <stdlib.h>
#include "netio_win.h"
#ifdef _MSC_VER
#pragma comment(lib, "ws2_32.lib")
#endif
#define CAP (1u << 20)

int main(int argc, char *argv[]) {
    if (argc != 2) { fprintf(stderr, "usage: %s PORT\n", argv[0]); return 1; }
    setvbuf(stdout, NULL, _IONBF, 0);
    WSADATA wsa;
    if (WSAStartup(MAKEWORD(2, 2), &wsa) != 0) { fprintf(stderr, "WSAStartup failed\n"); return 1; }
    SOCKET ls = socket(AF_INET, SOCK_STREAM, IPPROTO_TCP);
    if (ls == INVALID_SOCKET) { fprintf(stderr, "socket: %d\n", WSAGetLastError()); return 1; }
    struct sockaddr_in addr = {0};
    addr.sin_family = AF_INET;
    addr.sin_addr.s_addr = htonl(INADDR_LOOPBACK);
    addr.sin_port = htons((u_short)atoi(argv[1]));
    if (bind(ls, (struct sockaddr *)&addr, sizeof addr) == SOCKET_ERROR) { fprintf(stderr, "bind: %d\n", WSAGetLastError()); return 1; }
    if (listen(ls, 8) == SOCKET_ERROR) { fprintf(stderr, "listen: %d\n", WSAGetLastError()); return 1; }
    printf("listening on 127.0.0.1:%s\n", argv[1]);
    char *buf = malloc(CAP);
    if (!buf) return 1;
    for (;;) {
        SOCKET cs = accept(ls, NULL, NULL);
        if (cs == INVALID_SOCKET) continue;
        long len;
        while ((len = recv_msg(cs, buf, CAP)) > 0) {
            printf("message of %ld bytes\n", len);
            char reply[32];
            int rl = snprintf(reply, sizeof reply, "OK %ld", len);
            if (send_msg(cs, reply, (uint32_t)rl) < 0) break;
        }
        printf(len < 0 ? "bad or too long message: closing\n" : "client closed\n");
        closesocket(cs);
    }
}

In [ ]:
%%writefile lab3/win/msg_client_win.c
/* msg_client_win.c - Winsock port of msg_client.c: sends every argument before reading replies.
   Build:  gcc msg_client_win.c -o msg_client.exe -lws2_32
   Usage:  msg_client.exe HOST PORT MSG... */
#include <stdio.h>
#include <string.h>
#include "netio_win.h"
#ifdef _MSC_VER
#pragma comment(lib, "ws2_32.lib")
#endif

int main(int argc, char *argv[]) {
    if (argc < 4) { fprintf(stderr, "usage: %s HOST PORT MSG...\n", argv[0]); return 1; }
    WSADATA wsa;
    if (WSAStartup(MAKEWORD(2, 2), &wsa) != 0) { fprintf(stderr, "WSAStartup failed\n"); return 1; }
    struct addrinfo hints = {0}, *res, *ai;
    hints.ai_family = AF_UNSPEC; hints.ai_socktype = SOCK_STREAM;
    if (getaddrinfo(argv[1], argv[2], &hints, &res)) { fprintf(stderr, "getaddrinfo failed\n"); return 1; }
    SOCKET s = INVALID_SOCKET;
    for (ai = res; ai != NULL; ai = ai->ai_next) {
        s = socket(ai->ai_family, ai->ai_socktype, ai->ai_protocol);
        if (s == INVALID_SOCKET) continue;
        if (connect(s, ai->ai_addr, (int)ai->ai_addrlen) == 0) break;
        closesocket(s); s = INVALID_SOCKET;
    }
    freeaddrinfo(res);
    if (s == INVALID_SOCKET) { fprintf(stderr, "connect failed: error %d\n", WSAGetLastError()); return 1; }
    for (int i = 3; i < argc; i++)
        if (send_msg(s, argv[i], (uint32_t)strlen(argv[i])) < 0) { fprintf(stderr, "send: %d\n", WSAGetLastError()); return 1; }
    for (int i = 3; i < argc; i++) {
        char reply[64];
        long n = recv_msg(s, reply, sizeof reply - 1);
        if (n <= 0) { fprintf(stderr, "no reply\n"); break; }
        reply[n] = '\0';
        printf("%s -> %s\n", argv[i], reply);
    }
    closesocket(s);
    WSACleanup();
    return 0;
}

In [ ]:
WIN = os.path.join(LAB, 'win')
for prog in ['echo_client', 'echo_server', 'upper_server', 'msg_server', 'msg_client']:
    sh(f'cd {WIN} && x86_64-w64-mingw32-gcc -Wall -Wextra -O2 {prog}_win.c -o {prog}.exe -lws2_32 && echo "{prog}.exe built, no warnings"')
sh(f'cd {WIN} && file *.exe')

In [ ]:
# The most common build error of the Windows sheet: forgetting -lws2_32
sh(f'cd {WIN} && x86_64-w64-mingw32-gcc echo_server_win.c -o /tmp/broken.exe 2>&1 | grep -m3 "undefined reference"')

### Download and run on Windows
The cell below packs the sources and `.exe` files into `lab3_windows.zip`. On your PC: unzip, open PowerShell in the folder, and run `.\echo_server.exe 5000` in one window and `.\echo_client.exe 127.0.0.1 5000` in another (Tasks 2–3 of the Windows sheet).

In [ ]:
import zipfile
with zipfile.ZipFile('lab3_windows.zip', 'w', zipfile.ZIP_DEFLATED) as z:
    for f in sorted(os.listdir(WIN)):
        z.write(os.path.join(WIN, f), f'lab3_windows/{f}')
    for f in ('echo_server.py', 'echo_client.py'):
        z.write(os.path.join(LAB, 'python', f), f'lab3_windows/python/{f}')
print('lab3_windows.zip:', os.path.getsize('lab3_windows.zip'), 'bytes')
if IN_COLAB:
    from google.colab import files
    files.download('lab3_windows.zip')

### Optional: run the `.exe` files here with Wine
Wine runs Windows programs on Linux and implements Winsock on top of Linux sockets. Set `USE_WINE = True` (installation takes 2–4 minutes). A Windows client can then talk to a Linux server and vice versa — the protocol is the same, only the API differs.

In [ ]:
USE_WINE = False

if USE_WINE:
    if not (have('wine') or have('wine64')):
        sh(f'DEBIAN_FRONTEND=noninteractive {SUDO}apt-get -qq install -y wine wine64 > /dev/null 2>&1; echo installed', timeout=900)
    WINE = 'wine' if have('wine') else 'wine64'
    os.environ.update(WINEDEBUG='-all', WINEPREFIX='/tmp/wineprefix')
    WS = Bg(f'{WINE} ./echo_server.exe 5100', cwd=WIN, settle=40)        # the first start of Wine is slow
    sh(f'cd {WIN} && printf "hello from a Windows client\\n" | timeout 60 {WINE} ./echo_client.exe 127.0.0.1 5100', timeout=90)
    sh(f'cd {LAB} && ./blast 127.0.0.1 5100 300000')                    # Linux client → Windows server
    time.sleep(1); show('echo_server.exe (under Wine)', WS.output()); WS.stop()
else:
    print('Wine not used (set USE_WINE = True to try).')

### Questions that depend on Windows itself
* **Q4 (Windows)** — `connect` to a closed port fails with **10061** after about **2 s** on Windows: when the SYN is answered with RST, Windows retries the SYN twice more before giving up; Linux reports `ECONNREFUSED` on the first RST.
* **Q7 (Windows)** — without `htons`, port 5000 becomes **34835**, as on Linux: both run on little-endian x86 processors (see the computation below).
* **Q16 (Windows)** — on Windows, `SO_REUSEADDR` lets a *second* program bind a port already in use (even steal its traffic). `SO_EXCLUSIVEADDRUSE` forbids any other socket from binding the same address and port; servers should set it.

In [ ]:
print('Q7: 5000 without htons →', struct.unpack('!H', struct.pack('<H', 5000))[0])

### Task 5 (Windows) from PowerShell — optional
The sheet's PowerShell snippet uses .NET's `TcpClient`, which also exists in PowerShell 7 on Linux. Set `INSTALL_PWSH = True` to install it and send 300 000 bytes in one `Write` to our C echo server.

In [ ]:
INSTALL_PWSH = False

if INSTALL_PWSH and not have('pwsh'):
    ver = re.search(r'VERSION_ID="([\d.]+)"', open('/etc/os-release').read()).group(1)
    sh(f'wget -q https://packages.microsoft.com/config/ubuntu/{ver}/packages-microsoft-prod.deb -O /tmp/ms.deb && '
       f'{SUDO}dpkg -i /tmp/ms.deb > /dev/null && {SUDO}apt-get -qq update > /dev/null && '
       f'{SUDO}apt-get -qq install -y powershell > /dev/null; pwsh -Version', timeout=600)
PS = r"""
$c = New-Object System.Net.Sockets.TcpClient('127.0.0.1', 5000)
$s = $c.GetStream()
$data = [byte[]]::new(300000)
$s.Write($data, 0, $data.Length)          # one write of 300 000 bytes
$c.Client.Shutdown('Send')                # like shutdown(SHUT_WR)
$buf = [byte[]]::new(65536); while ($s.Read($buf, 0, $buf.Length) -gt 0) { }
$c.Close()
"""
if have('pwsh'):
    S = Bg('./echo_server 5000')
    open('/tmp/blast.ps1', 'w').write(PS); sh('pwsh -NoProfile -File /tmp/blast.ps1', timeout=60)
    time.sleep(0.5); show('echo_server', S.output()); S.stop()
else:
    print('PowerShell not installed (set INSTALL_PWSH = True to try).')

**✍️ Your answers** (double-click to edit)

- **Q3 (Windows) — the Winsock differences in the code; which one fails immediately if forgotten?:** 
- **Q11–Q12 (Windows) — line framing on Windows:** 
- **Q13–Q14 (Windows) — length prefix; file size check with (Get-Item file).Length:** 

---
### Next week
Lecture 4 continues with the Web: cookies, caching, HTTP/2, HTTP/3 and content distribution networks. Laboratory 4 moves to UDP sockets and byte order.